In [0]:
# 1. catálogo workspace
spark.sql("USE CATALOG workspace")

# 2. schema/banco de dados
spark.sql("CREATE SCHEMA IF NOT EXISTS db_viagens_gov")

# 3. Volume oficial para receber os arquivos CSV
spark.sql("CREATE VOLUME IF NOT EXISTS db_viagens_gov.viagens_gov")

print("Schema 'workspace.db_viagens_gov' e Volume criados com sucesso!")

Schema 'workspace.db_viagens_gov' e Volume criados com sucesso!


In [0]:
import os
from pyspark.sql.functions import current_timestamp, col

volume_path = "/Volumes/workspace/db_viagens_gov/viagens_gov"

mapeamento_arquivos = {
    "bronze_viagens": "2025_Viagem.csv",
    "bronze_trechos": "2025_Trecho.csv",
    "bronze_passagens": "2025_Passagem.csv",
    "bronze_pagamentos": "2025_Pagamento.csv"
}

for tabela_destino, nome_arquivo in mapeamento_arquivos.items():
    caminho_spark = os.path.join(volume_path, nome_arquivo)
    
    print(f"Lendo: {nome_arquivo} ...")
    
    # Leitura do CSV 
    df_raw = spark.read \
        .option("header", "true") \
        .option("delimiter", ";") \
        .option("encoding", "Windows-1252") \
        .csv(caminho_spark)
    
    # Metadados de auditoria e controle de linhagem
    df_bronze = df_raw \
        .withColumn("_data_ingestao", current_timestamp()) \
        .withColumn("_fonte_origem", col("_metadata.file_path"))
    
    # Gravação na camada Bronze 
    df_bronze.write \
        .format("delta") \
        .mode("overwrite") \
        .option("delta.columnMapping.mode", "name") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"workspace.db_viagens_gov.{tabela_destino}")
    
    print(f"Tabela 'workspace.db_viagens_gov.{tabela_destino}' criada com sucesso!\n")

Lendo: 2025_Viagem.csv ...
Tabela 'workspace.db_viagens_gov.bronze_viagens' criada com sucesso!

Lendo: 2025_Trecho.csv ...
Tabela 'workspace.db_viagens_gov.bronze_trechos' criada com sucesso!

Lendo: 2025_Passagem.csv ...
Tabela 'workspace.db_viagens_gov.bronze_passagens' criada com sucesso!

Lendo: 2025_Pagamento.csv ...
Tabela 'workspace.db_viagens_gov.bronze_pagamentos' criada com sucesso!



In [0]:
display(spark.sql("SELECT * FROM workspace.db_viagens_gov.bronze_viagens LIMIT 20"))

Identificador do processo de viagem,Número da Proposta (PCDP),Situação,Viagem Urgente,Justificativa Urgência Viagem,Código do órgão superior,Nome do órgão superior,Código órgão solicitante,Nome órgão solicitante,CPF viajante,Nome,Cargo,Função,Descrição Função,Período - Data de início,Período - Data de fim,Destinos,Motivo,Valor diárias,Valor passagens,Valor devolução,Valor outros gastos,_data_ingestao,_fonte_origem
0000000000021378576,002303/25,Realizada,NÃO,Sem informação,26000,Ministério da Educação,26406,"Instituto Federal de Educação, Ciência e Tecnologia do Espírito Santo",***.613.727-**,LUIZA SANTOS ALVES,PROFESSOR ENS BASICO TECN TECNOLOGICO,-1,Sem informação,17/09/2025,19/09/2025,Vitória/ES,"Participar do 'Seminário 'Gênero, sexualidade(s) e outros marcadores da diferença: o que pode um corpo quando fracassa?', ofertado pelo(a) Universidade Federal do Espírito Santo, com apoio do Instituto Federal do Espírito Santo, na modalidade Presencial, a ser realizado em Vitória /ES no período de 17 a 19/09/2025, com carga horária total de 20 horas.","0,00","0,00","0,00","0,00",2026-09-26T19:15:01.259Z,dbfs:/Volumes/workspace/db_viagens_gov/viagens_gov/2025_Viagem.csv
0000000000021378577,050130/25,Realizada,NÃO,Sem informação,52000,Ministério da Defesa,52111,Comando da Aeronáutica,***.809.992-**,DIONISIO JOSE VIEGAS DA SILVA JUNIOR,null,Suboficial,Suboficial,20/10/2025,24/10/2025,Ipameri/GO,SC: 04.05.00.,"1440,00","0,00","0,00","0,00",2026-09-26T19:15:01.259Z,dbfs:/Volumes/workspace/db_viagens_gov/viagens_gov/2025_Viagem.csv
0000000000021378578,043257/25,Realizada,NÃO,Sem informação,30000,Ministério da Justiça e Segurança Pública,30000,Ministério da Justiça e Segurança Pública - Unidades com vínculo direto,***.310.209-**,MARCO ROBERTO ALVES DOS REIS,null,-1,Sem informação,08/10/2025,15/10/2025,Querência do Norte/PR,"Operação Protetor das Divisas e Fronteiras, no Estado do Paraná - período de 01/10/2025 a 31/10/2025 - Processo (08020.009130/2025-48) - não constituído de MICRORREGIÃO ou MUNICÍPIOS LIMÍTROFES, conforme explicitado no Diárias e Passagens Nacionais - Coletivo 32929263 e autorizado pelo Despacho 1069 (32929267) Centro de Custo: 111I04","2285,04","0,00","0,00","0,00",2026-09-26T19:15:01.259Z,dbfs:/Volumes/workspace/db_viagens_gov/viagens_gov/2025_Viagem.csv
0000000000021378579,043289/25,Realizada,NÃO,Sem informação,30000,Ministério da Justiça e Segurança Pública,30000,Ministério da Justiça e Segurança Pública - Unidades com vínculo direto,***.962.849-**,WILMAR GUILHERME DE OLIVEIRA GOMES,null,-1,Sem informação,15/10/2025,22/10/2025,Querência do Norte/PR,"Operação Protetor das Divisas e Fronteiras, no Estado do Paraná - período de 01/10/2025 a 31/10/2025 - Processo (08020.009130/2025-48) - não constituído de MICRORREGIÃO ou MUNICÍPIOS LIMÍTROFES, conforme explicitado no Diárias e Passagens Nacionais - Coletivo 32929263 e autorizado pelo Despacho 1069 (32929267) Centro de Custo: 111I04","2285,04","0,00","0,00","0,00",2026-09-26T19:15:01.259Z,dbfs:/Volumes/workspace/db_viagens_gov/viagens_gov/2025_Viagem.csv
0000000000021378580,004017/25,Realizada,NÃO,Sem informação,68000,Ministério de Portos e Aeroportos,62201,Agência Nacional de Aviação Civil,***.595.436-**,VINICIUS BRETAS QUINTAO,ESP EM REGULACAO DE AVIACAO CIVIL,-1,Sem informação,15/10/2025,21/10/2025,Stavanger/Noruega,"CCC: F0501 - [00874/2025/SPO] @ Ativ. 1 # Tipo de Ativ.: SPL - Fiscalização - Vigilância (programada) # Código de Atividade Planejada: CTAC-503 # Local: Stavanger - Noruega # Início: 17/10/25 09:40 # Término: 19/10/25 21:20 # Descrição: Realizar o acompanhamento de treinamento inicial para o modelo H145 e IFRH. Dia 17: Entre 13:00h e 16:30h, acompanhamento da 1ª sessão de um treinamento inicial do H145/IFRH. Entre 17:20h e 21:00h, acompanhamento da 2ª sessão de um treinamento inicial do H145/IFRH. Entre 21:30h e 01:00h, acompanhamento da 5ª sessão de outro treinamento inicial do H145/IFRH. Dia 18: Entre 13:00h e 16:30h, acompanhamento da 3ª sessão de um 

In [0]:
from pyspark.sql.functions import col, regexp_replace, to_date

# 1. SILVER VIAGENS (Limpeza, Tipagem e Renomeação)
df_bronze_viagens = spark.table("workspace.db_viagens_gov.bronze_viagens")

df_silver_viagens = df_bronze_viagens \
    .withColumnRenamed("Identificador do processo de viagem", "id_viagem") \
    .withColumnRenamed("Número da Proposta (PCDP)", "numero_proposta") \
    .withColumnRenamed("Código do órgão superior", "cod_orgao_superior") \
    .withColumnRenamed("Nome do órgão superior", "orgao_superior") \
    .withColumnRenamed("Código órgão solicitante", "cod_orgao_solicitante") \
    .withColumnRenamed("Nome órgão solicitante", "orgao_solicitante") \
    .withColumnRenamed("CPF viajante", "cpf_viajante") \
    .withColumnRenamed("Nome", "nome_viajante") \
    .withColumnRenamed("Viagem Urgente", "indicador_urgente") \
    .withColumnRenamed("Justificativa Urgência Viagem", "justificativa_urgencia") \
    .withColumn("dt_inicio_afastamento", to_date(col("Período - Data de início"), "dd/MM/yyyy")) \
    .withColumn("dt_fim_afastamento", to_date(col("Período - Data de fim"), "dd/MM/yyyy")) \
    .withColumn("val_diarias", regexp_replace(regexp_replace(col("Valor diárias"), "\\.", ""), ",", ".").cast("decimal(15,2)")) \
    .withColumn("val_passagens", regexp_replace(regexp_replace(col("Valor passagens"), "\\.", ""), ",", ".").cast("decimal(15,2)")) \
    .withColumn("val_devolucao", regexp_replace(regexp_replace(col("Valor devolução"), "\\.", ""), ",", ".").cast("decimal(15,2)")) \
    .withColumn("val_outros_gastos", regexp_replace(regexp_replace(col("Valor outros gastos"), "\\.", ""), ",", ".").cast("decimal(15,2)")) \
    .dropDuplicates(["id_viagem"])

colunas_silver_viagens = [
    "id_viagem", "numero_proposta", "cod_orgao_superior", "orgao_superior",
    "cod_orgao_solicitante", "orgao_solicitante", "cpf_viajante", "nome_viajante",
    "indicador_urgente", "justificativa_urgencia", "dt_inicio_afastamento",
    "dt_fim_afastamento", "val_diarias", "val_passagens", "val_devolucao", "val_outros_gastos"
]

df_silver_viagens.select(*colunas_silver_viagens).write \
    .format("delta") \
    .mode("overwrite") \
    .option("delta.columnMapping.mode", "name") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.db_viagens_gov.silver_viagens")

print("Tabela 'workspace.db_viagens_gov.silver_viagens' criada com sucesso!")

# 2. SILVER PASSAGENS (Limpeza, Padronização e Tipagem)
df_bronze_passagens = spark.table("workspace.db_viagens_gov.bronze_passagens")

df_silver_passagens = df_bronze_passagens \
    .withColumnRenamed("Identificador do processo de viagem", "id_viagem") \
    .withColumnRenamed("Número da Proposta (PCDP)", "numero_proposta") \
    .withColumnRenamed("Meio de transporte", "meio_transporte") \
    .withColumnRenamed("País - Origem ida", "pais_origem_ida") \
    .withColumnRenamed("UF - Origem ida", "uf_origem_ida") \
    .withColumnRenamed("Cidade - Origem ida", "cidade_origem_ida") \
    .withColumnRenamed("País - Destino ida", "pais_destino_ida") \
    .withColumnRenamed("UF - Destino ida", "uf_destino_ida") \
    .withColumnRenamed("Cidade - Destino ida", "cidade_destino_ida") \
    .withColumnRenamed("País - Origem volta", "pais_origem_volta") \
    .withColumnRenamed("UF - Origem volta", "uf_origem_volta") \
    .withColumnRenamed("Cidade - Origem volta", "cidade_origem_volta") \
    .withColumnRenamed("Pais - Destino volta", "pais_destino_volta") \
    .withColumnRenamed("UF - Destino volta", "uf_destino_volta") \
    .withColumnRenamed("Cidade - Destino volta", "cidade_destino_volta") \
    .withColumn("val_passagem", regexp_replace(regexp_replace(col("Valor da passagem"), "\\.", ""), ",", ".").cast("decimal(15,2)")) \
    .withColumn("val_taxa_servico", regexp_replace(regexp_replace(col("Taxa de serviço"), "\\.", ""), ",", ".").cast("decimal(15,2)")) \
    .withColumn("dt_emissao_compra", to_date(col("Data da emissão/compra"), "dd/MM/yyyy"))

colunas_silver_passagens = [
    "id_viagem", "numero_proposta", "meio_transporte", 
    "pais_origem_ida", "uf_origem_ida", "cidade_origem_ida",
    "pais_destino_ida", "uf_destino_ida", "cidade_destino_ida",
    "pais_origem_volta", "uf_origem_volta", "cidade_origem_volta",
    "pais_destino_volta", "uf_destino_volta", "cidade_destino_volta",
    "val_passagem", "val_taxa_servico", "dt_emissao_compra"
]

df_silver_passagens.select(*colunas_silver_passagens).write \
    .format("delta") \
    .mode("overwrite") \
    .option("delta.columnMapping.mode", "name") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.db_viagens_gov.silver_passagens")

print("Tabela 'workspace.db_viagens_gov.silver_passagens' criada com sucesso!")

Tabela 'workspace.db_viagens_gov.silver_viagens' criada com sucesso!
Tabela 'workspace.db_viagens_gov.silver_passagens' criada com sucesso!


In [0]:
display(spark.sql("""
    SELECT 
        v.id_viagem, 
        v.orgao_superior, 
        v.indicador_urgente, 
        p.meio_transporte,
        p.cidade_destino_ida,
        v.val_diarias, 
        p.val_passagem
    FROM workspace.db_viagens_gov.silver_viagens v
    LEFT JOIN workspace.db_viagens_gov.silver_passagens p ON v.id_viagem = p.id_viagem
    LIMIT 5000
"""))

id_viagem,orgao_superior,indicador_urgente,meio_transporte,cidade_destino_ida,val_diarias,val_passagem
0000000000021247285,Ministério da Defesa,NÃO,Aéreo,Rio de Janeiro,0.00,667.97
0000000000021247285,Ministério da Defesa,NÃO,Aéreo,Brasília,0.00,469.86
0000000000021258704,Ministério de Portos e Aeroportos,SIM,Aéreo,Rio de Janeiro,1280.25,1975.09
0000000000021258704,Ministério de Portos e Aeroportos,SIM,Aéreo,Campinas,1280.25,360.68
0000000000021258704,Ministério de Portos e Aeroportos,SIM,Aéreo,Rio de Janeiro,1280.25,356.89
0000000000021266050,Ministério da Educação,SIM,Aéreo,Vitória,963.65,689.72
0000000000021266050,Ministério da Educação,SIM,Aéreo,Campo Grande,963.65,1204.24
0000000000021267917,Presidência da República,SIM,Aéreo,Bogotá,0.00,1012.28
0000000000021271334,Ministério da Cultura,SIM,Aéreo,São Paulo,904.10,2452.65
0000000000021271334,Ministério da Cultura,SIM,Aéreo,Brasília,904.10,2312.62


In [0]:
%sql
-- 1. Dimensão Órgão
CREATE TABLE IF NOT EXISTS workspace.db_viagens_gov.dim_orgao (
    sk_orgao BIGINT NOT NULL,
    cod_orgao_superior INT,
    orgao_superior VARCHAR(255),
    cod_orgao_solicitante INT,
    orgao_solicitante VARCHAR(255)
) USING DELTA;

-- 2. Dimensão Viajante
CREATE TABLE IF NOT EXISTS workspace.db_viagens_gov.dim_viajante (
    sk_viajante BIGINT NOT NULL,
    cpf_viajante VARCHAR(14),
    nome_viajante VARCHAR(255)
) USING DELTA;

-- 3. Dimensão Tempo
CREATE TABLE IF NOT EXISTS workspace.db_viagens_gov.dim_tempo (
    sk_tempo INT NOT NULL,
    data_completa DATE NOT NULL,
    ano INT NOT NULL,
    mes INT NOT NULL,
    nome_mes VARCHAR(20) NOT NULL,
    trimestre INT NOT NULL,
    semestre INT NOT NULL,
    dia_semana VARCHAR(20) NOT NULL,
    eh_fim_semana VARCHAR(3) NOT NULL
) USING DELTA;

-- 4. Dimensão Meio de Transporte
CREATE TABLE IF NOT EXISTS workspace.db_viagens_gov.dim_meio_transporte (
    sk_meio_transporte INT NOT NULL,
    meio_transporte VARCHAR(50) NOT NULL
) USING DELTA;

-- 5. Fato Viagens
CREATE TABLE IF NOT EXISTS workspace.db_viagens_gov.fato_viagens (
    id_viagem VARCHAR(50) NOT NULL,
    numero_proposta VARCHAR(50),
    sk_orgao BIGINT NOT NULL,
    sk_viajante BIGINT NOT NULL,
    sk_tempo_inicio INT NOT NULL,
    sk_tempo_fim INT NOT NULL,
    indicador_urgente VARCHAR(10),
    qtd_dias_viagem INT,
    val_diarias DECIMAL(15,2) NOT NULL,
    val_passagens DECIMAL(15,2) NOT NULL,
    val_devolucao DECIMAL(15,2),
    val_outros_gastos DECIMAL(15,2),
    val_total_viagem DECIMAL(15,2) NOT NULL
) USING DELTA;

-- 6. Fato Passagens
CREATE TABLE IF NOT EXISTS workspace.db_viagens_gov.fato_passagens (
    id_viagem VARCHAR(50) NOT NULL,
    sk_orgao BIGINT NOT NULL,
    sk_viajante BIGINT NOT NULL,
    sk_meio_transporte INT NOT NULL,
    sk_tempo_emissao INT NOT NULL,
    uf_origem CHAR(2),
    cidade_origem VARCHAR(150),
    uf_destino CHAR(2),
    cidade_destino VARCHAR(150),
    val_passagem DECIMAL(15,2) NOT NULL,
    val_taxa_servico DECIMAL(15,2),
    val_total_bilhete DECIMAL(15,2) NOT NULL    
) USING DELTA;

In [0]:
# CÉLULA DE CARGA DA CAMADA GOLD (ETL)
from pyspark.sql.functions import (
    col, monotonically_increasing_id, year, month, 
    date_format, dayofweek, datediff, coalesce, lit
)

# 1. CARGA DA DIM_ORGAO
df_silver_v = spark.table("workspace.db_viagens_gov.silver_viagens")

df_dim_orgao = df_silver_v.select(
    "orgao_superior"
).distinct().withColumn("sk_orgao", monotonically_increasing_id() + 1)

df_dim_orgao.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.db_viagens_gov.dim_orgao")

# 2. CARGA DA DIM_MEIO_TRANSPORTE
df_silver_p = spark.table("workspace.db_viagens_gov.silver_passagens")

df_dim_transporte = df_silver_p.select("meio_transporte").filter(col("meio_transporte").isNotNull()).distinct() \
    .withColumn("sk_meio_transporte", monotonically_increasing_id() + 1)

df_dim_transporte.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.db_viagens_gov.dim_meio_transporte")

# 3. CARGA DA DIM_VIAJANTE (Buscando diretamente da tabela silver_passagens caso exista o campo lá)
# Se a silver_passagens não tiver o campo nome_viajante/cpf_viajante, comente este bloco.
try:
    df_dim_viajante = df_silver_p.select("cpf_viajante", "nome_viajante").distinct() \
        .withColumn("sk_viajante", monotonically_increasing_id() + 1)

    df_dim_viajante.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
        .saveAsTable("workspace.db_viagens_gov.dim_viajante")
    has_viajante = True
except Exception:
    has_viajante = False

# 4. CARGA DA DIM_TEMPO 
df_datas = df_silver_v.select(col("dt_inicio_afastamento").alias("data")).union(
    df_silver_v.select(col("dt_fim_afastamento").alias("data"))
).filter(col("data").isNotNull()).distinct()

df_dim_tempo = df_datas.select(
    date_format(col("data"), "yyyyMMdd").cast("int").alias("sk_tempo"),
    col("data").alias("data_completa"),
    year(col("data")).alias("ano"),
    month(col("data")).alias("mes"),
    date_format(col("data"), "MMMM").alias("nome_mes"),
    ((month(col("data")) - 1) / 3 + 1).cast("int").alias("trimestre"),
    ((month(col("data")) - 1) / 6 + 1).cast("int").alias("semestre"),
    date_format(col("data"), "EEEE").alias("dia_semana"),
    date_format(col("data"), "E").alias("eh_fim_semana")
)

df_dim_tempo.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.db_viagens_gov.dim_tempo")

# 5. CARGA DA FATO_VIAGENS
df_fato_viagens = df_silver_v \
    .join(df_dim_orgao, ["orgao_superior"], "left") \
    .withColumn("sk_tempo_inicio", date_format(col("dt_inicio_afastamento"), "yyyyMMdd").cast("int")) \
    .withColumn("sk_tempo_fim", date_format(col("dt_fim_afastamento"), "yyyyMMdd").cast("int")) \
    .withColumn("qtd_dias_viagem", datediff(col("dt_fim_afastamento"), col("dt_inicio_afastamento"))) \
    .withColumn("val_total_viagem", 
                coalesce(col("val_diarias"), lit(0)) + 
                coalesce(col("val_passagens"), lit(0)) + 
                coalesce(col("val_outros_gastos"), lit(0)) - 
                coalesce(col("val_devolucao"), lit(0))) \
    .select(
        "id_viagem", "sk_orgao",
        "sk_tempo_inicio", "sk_tempo_fim", "indicador_urgente", "situacao",
        "qtd_dias_viagem", "val_diarias", "val_passagens",
        "val_devolucao", "val_outros_gastos", "val_total_viagem"
    )

df_fato_viagens.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.db_viagens_gov.fato_viagens")

print("Todas as Dimensões e a Tabela Fato_Viagens foram povoadas com sucesso na Camada Gold!")

Todas as Dimensões e a Tabela Fato_Viagens foram povoadas com sucesso na Camada Gold!


In [0]:
spark.table("workspace.db_viagens_gov.silver_viagens").printSchema()

root
 |-- id_viagem: string (nullable = true)
 |-- orgao_superior: string (nullable = true)
 |-- indicador_urgente: string (nullable = true)
 |-- situacao: string (nullable = true)
 |-- dt_inicio_afastamento: date (nullable = true)
 |-- dt_fim_afastamento: date (nullable = true)
 |-- val_diarias: double (nullable = true)
 |-- val_passagens: double (nullable = true)
 |-- val_devolucao: double (nullable = true)
 |-- val_outros_gastos: double (nullable = true)



In [0]:
%sql
-- 1. QUAIS ÓRGÃOS SUPERIORES TIVERAM OS MAIORES GASTOS TOTAIS?
SELECT 
    orgao_superior,
    COUNT(DISTINCT id_viagem) AS total_viagens,
    ROUND(SUM(val_diarias), 2) AS total_diarias_brl,
    ROUND(SUM(val_passagens), 2) AS total_passagens_brl,
    ROUND(SUM(COALESCE(val_diarias, 0) + COALESCE(val_passagens, 0) + COALESCE(val_outros_gastos, 0) - COALESCE(val_devolucao, 0)), 2) AS gasto_total_real_brl
FROM workspace.db_viagens_gov.silver_viagens
WHERE orgao_superior IS NOT NULL
GROUP BY orgao_superior
ORDER BY gasto_total_real_brl DESC
LIMIT 10;

orgao_superior,total_viagens,total_diarias_brl,total_passagens_brl,gasto_total_real_brl
Ministério da Justiça e Segurança Pública,174001,709235922.47,118329034.22,824242072.46
Ministério da Defesa,135133,192583524.90,155290892.02,348499487.00
Ministério da Educação,179472,178392822.85,151378551.20,330347633.70
Ministério do Meio Ambiente e Mudança do Clima,48606,89204142.41,45429717.83,133896403.80
Ministério da Saúde,19445,39936708.80,46204030.28,86129942.49
Ministério da Previdência Social,18064,58690169.07,19848376.48,79319982.65
Ministério da Fazenda,26295,40104449.91,31347622.54,72030703.36
Ministério do Desenvolvimento Agrário e Agricultura Familiar,26072,41278072.79,23179242.29,64111350.17
Ministério dos Povos Indígenas,12320,41418954.84,17609109.36,58740190.75
Ministério das Relações Exteriores,4152,15502963.63,33548210.73,48812275.24


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- 2. QUAL O IMPACTO FINANCEIRO DE VIAGENS CLASSIFICADAS COMO URGENTE ?
SELECT 
    COALESCE(indicador_urgente, 'Não Informado') AS indicador_urgente,
    COUNT(DISTINCT id_viagem) AS qtd_viagens,
    ROUND(SUM(
        COALESCE(val_diarias, 0) + 
        COALESCE(val_passagens, 0) + 
        COALESCE(val_outros_gastos, 0) - 
        COALESCE(val_devolucao, 0)
    ), 2) AS gasto_total_real_brl,
    ROUND(AVG(
        COALESCE(val_diarias, 0) + 
        COALESCE(val_passagens, 0) + 
        COALESCE(val_outros_gastos, 0) - 
        COALESCE(val_devolucao, 0)
    ), 2) AS custo_medio_por_viagem_brl
FROM workspace.db_viagens_gov.silver_viagens
GROUP BY indicador_urgente
ORDER BY gasto_total_real_brl DESC;

indicador_urgente,qtd_viagens,gasto_total_real_brl,custo_medio_por_viagem_brl
SIM,468333,1520692402.23,3247.03
NÃO,346059,999801160.75,2889.11


In [0]:
from pyspark.sql import functions as F

CAMINHO_ARQUIVO = "/Volumes/workspace/db_viagens_gov/viagens_gov/2025_Viagem.csv"

df_raw = spark.read \
    .option("header", "true") \
    .option("delimiter", ";") \
    .option("encoding", "ISO-8859-1") \
    .csv(CAMINHO_ARQUIVO)

df_silver_clean = df_raw \
    .withColumn("id_viagem", F.col("Identificador do processo de viagem")) \
    .withColumn("orgao_superior", F.col("Nome do órgão superior")) \
    .withColumn("indicador_urgente", F.col("Viagem Urgente")) \
    .withColumn("situacao", F.col("Situação")) \
    .withColumn("dt_inicio_afastamento", F.to_date(F.col("Período - Data de início"), "dd/MM/yyyy")) \
    .withColumn("dt_fim_afastamento", F.to_date(F.col("Período - Data de fim"), "dd/MM/yyyy")) \
    .withColumn("val_diarias", F.regexp_replace(F.col("Valor diárias"), ",", ".").cast("double")) \
    .withColumn("val_passagens", F.regexp_replace(F.col("Valor passagens"), ",", ".").cast("double")) \
    .withColumn("val_devolucao", F.regexp_replace(F.col("Valor devolução"), ",", ".").cast("double")) \
    .withColumn("val_outros_gastos", F.regexp_replace(F.col("Valor outros gastos"), ",", ".").cast("double")) \
    .filter(F.year(F.col("dt_inicio_afastamento")) == 2025) \
    .dropDuplicates(["id_viagem"]) \
    .select(
        "id_viagem", 
        "orgao_superior", 
        "indicador_urgente", 
        "situacao",
        "dt_inicio_afastamento", 
        "dt_fim_afastamento", 
        "val_diarias", 
        "val_passagens", 
        "val_devolucao", 
        "val_outros_gastos"
    )

df_silver_clean.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.columnMapping.mode", "name") \
    .saveAsTable("workspace.db_viagens_gov.silver_viagens")

print("Tabela silver_viagens recriada com a coluna de situação!")

Tabela silver_viagens recriada com a coluna de situação!


In [0]:
%sql
SHOW VOLUMES IN workspace.db_viagens_gov;

database,volume_name
db_viagens_gov,viagens_gov


In [0]:
%sql
-- 3. DISTRIBUIÇÃO MENSAL DE GASTOS AO LONGO DO ANO (SAZONALIDADE)
SELECT 
    MONTH(dt_inicio_afastamento) AS mes_num,
    CASE MONTH(dt_inicio_afastamento)
        WHEN 1 THEN 'JANEIRO'
        WHEN 2 THEN 'FEVEREIRO'
        WHEN 3 THEN 'MARÇO'
        WHEN 4 THEN 'ABRIL'
        WHEN 5 THEN 'MAIO'
        WHEN 6 THEN 'JUNHO'
        WHEN 7 THEN 'JULHO'
        WHEN 8 THEN 'AGOSTO'
        WHEN 9 THEN 'SETEMBRO'
        WHEN 10 THEN 'OUTUBRO'
        WHEN 11 THEN 'NOVEMBRO'
        WHEN 12 THEN 'DEZEMBRO'
    END AS meses,
    COUNT(DISTINCT id_viagem) AS qtd_viagens,
    ROUND(SUM(COALESCE(val_diarias, 0) + COALESCE(val_passagens, 0) - COALESCE(val_devolucao, 0)), 2) AS vl_total_viagens_pipeline
FROM workspace.db_viagens_gov.silver_viagens
WHERE dt_inicio_afastamento IS NOT NULL
  AND YEAR(dt_inicio_afastamento) = 2025
  AND LOWER(orgao_superior) NOT LIKE '%justiça%' 
GROUP BY MONTH(dt_inicio_afastamento)
ORDER BY mes_num ASC;

mes_num,meses,qtd_viagens,vl_total_viagens_pipeline
1,JANEIRO,19850,5.34394804E7
2,FEVEREIRO,34162,8.448322008E7
3,MARÇO,48907,1.3606319439E8
4,ABRIL,47742,1.1590833756E8
5,MAIO,58053,1.5311936975E8
6,JUNHO,58078,1.5553439861E8
7,JULHO,46120,1.1748866215E8
8,AGOSTO,62844,1.6574386435E8
9,SETEMBRO,74275,1.9768597393E8
10,OUTUBRO,69715,1.9121245466E8


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- 4. TOP 10 DESTINOS COM MAIOR CONSUMO DE ORÇAMENTO (PASSAGENS + DIÁRIAS) 
SELECT 
    UPPER(TRIM(cidade_destino_ida)) AS cidade_destino,
    UPPER(TRIM(uf_destino_ida)) AS uf_destino,
    COUNT(DISTINCT id_viagem) AS total_viagens_associadas,
    ROUND(SUM(COALESCE(val_passagem, 0)), 2) AS total_gasto_passagens_brl
FROM workspace.db_viagens_gov.silver_passagens
WHERE cidade_destino_ida IS NOT NULL 
  AND TRIM(cidade_destino_ida) != ''
GROUP BY 
    UPPER(TRIM(cidade_destino_ida)),
    UPPER(TRIM(uf_destino_ida))
ORDER BY total_gasto_passagens_brl DESC
LIMIT 10;


cidade_destino,uf_destino,total_viagens_associadas,total_gasto_passagens_brl
BRASÍLIA,DISTRITO FEDERAL,117996,232484257.17
RIO DE JANEIRO,RIO DE JANEIRO,39256,64893360.20
SÃO PAULO,SÃO PAULO,27507,40831794.08
MANAUS,AMAZONAS,14516,28186615.48
BELÉM,PARÁ,11470,21624999.90
FORTALEZA,CEARÁ,9991,19835768.08
RECIFE,PERNAMBUCO,11901,19464772.95
PORTO ALEGRE,RIO GRANDE DO SUL,11440,19263644.33
BOA VISTA,RORAIMA,6218,16425054.92
SALVADOR,BAHIA,9935,15053861.26


Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql import functions as F

print("SUÍTE DE TESTES DE ACURÁCIA E QUALIDADE DE DADOS")

df_silver = spark.table("workspace.db_viagens_gov.silver_viagens")
total_rows = df_silver.count()

# Teste 1: Unicidade das Chaves Primárias (Ausência de Duplicatas)
distinct_ids = df_silver.select("id_viagem").distinct().count()
test_unicity = (total_rows == distinct_ids)
status_t1 = "APROVADO" if test_unicity else "REPROVADO"
print(f"1. Teste de Unicidade (id_viagem): {status_t1} ({distinct_ids}/{total_rows})")

# Teste 2: Integridade de Valores Negativos Unjustified
neg_diarias = df_silver.filter(F.col("val_diarias") < 0).count()
neg_passagens = df_silver.filter(F.col("val_passagens") < 0).count()
test_non_negative = (neg_diarias == 0 and neg_passagens == 0)
status_t2 = "APROVADO" if test_non_negative else "REPROVADO"
print(f"2. Teste de Não-Negatividade (Valores): {status_t2} (Violações: Diárias={neg_diarias}, Passagens={neg_passagens})")

# Teste 3: Completude da Chave de Origem/Órgão
null_orgaos = df_silver.filter(F.col("orgao_superior").isNull()).count()
pct_orgao_null = (null_orgaos / total_rows) * 100
test_completeness = pct_orgao_null == 0
status_t3 = "APROVADO" if test_completeness else f"ATENÇÃO ({pct_orgao_null:.2f}% nulos)"
print(f"3. Teste de Completude (orgao_superior): {status_t3}")

# Teste 4: Consistência de Datas (Início <= Fim)
date_inconsistencies = df_silver.filter(F.col("dt_fim_afastamento") < F.col("dt_inicio_afastamento")).count()
test_dates = (date_inconsistencies == 0)
status_t4 = "APROVADO" if test_dates else f"ATENÇÃO ({date_inconsistencies} registros inconsistentes)"
print(f"4. Teste de Consistência Temporal (Data Fim >= Data Início): {status_t4}")

# Teste 5: Verificação de Outliers / Erros de Escala Decimal
max_diaria = df_silver.select(F.max("val_diarias")).collect()[0][0]
test_outlier = max_diaria < 1000000  # Nenhuma diária individual deve exceder R$ 1 milhão
status_t5 = "APROVADO" if test_outlier else "REPROVADO (Possível erro de vírgula/ponto)"
print(f"5. Teste de Limites Numéricos (Max Diária < R$ 1Mi): {status_t5} (Maior valor: R$ {max_diaria:,.2f})")

SUÍTE DE TESTES DE ACURÁCIA E QUALIDADE DE DADOS
1. Teste de Unicidade (id_viagem): APROVADO (814392/814392)
2. Teste de Não-Negatividade (Valores): APROVADO (Violações: Diárias=0, Passagens=0)
3. Teste de Completude (orgao_superior): APROVADO
4. Teste de Consistência Temporal (Data Fim >= Data Início): APROVADO
5. Teste de Limites Numéricos (Max Diária < R$ 1Mi): APROVADO (Maior valor: R$ 213,987.50)
